# Smoke Test: Phone Idle-Disconnect "Time Lost / Insert Coin" Fix — Theory Validation

**Rule for this session: validate BEFORE patching anything.**

This notebook simulates the exact RouterOS 6.49.17 hotspot behavior observed on the live hAP Lite (10.0.0.1) and the login page behavior captured from `C:\Users\rochiegwapo\Desktop\hotspot`, then tests the proposed fixes **in theory** before we flash/change anything.

Observed production facts baked into the model:
1. The user record `limit-uptime` does **not** decrement natively during a session (that is why ARDS23 existed). Record stays at the FULL original limit.
2. An active session's true remaining time = `session-budget − time elapsed while active`.
3. `keepalive-timeout` (profile `coin-hotspot` = **2m**) kills the session when the phone stops talking (~phone sleep), so the captive portal re-logs the phone.
4. On landing at `login.html`, `core.js` auto-login ONLY fires when the router serves `/data/<MAC>.txt` (content format `voucher#date`, parsed with `.split("#")[0]`). The live router's `data/` folder holds ONE stale file from July — our ESP32 firmware never writes these files (the old NodeMCU did). So returning phones get the plain INSERT COIN page.
5. Remaining-time persistence across reconnects requires something to sync the running session's remaining time back into the user record (candidate: 1-minute scheduler that only writes while the session is active, so it can never collide with the ESP's kick-first crediting).

Candidate fixes under test:
- **FIX A (ESP firmware):** after a successful credit, ESP writes `/data/<MAC>.txt` via RouterOS API `/file/add contents=` → login page auto-logs returning phones.
- **FIX B (Router profile):** `keepalive-timeout` 2m → 15m so sleeping phones survive short naps without re-login.
- **FIX C (Router scheduler, 1m):** sync active `session-time-left` → `limit-uptime` so a reconnect restores the EXACT remaining time (no reset, no loss).

In [ ]:
# Cell 2 - The simulated component (RouterOS hotspot + ESP bridge + login page)
# Pure stdlib, deterministic. Semantics mirror the LIVE box observed this week.

class Hotspot:
    """Models one customer journey: credit -> surf -> phone nap -> reconnect."""

    def __init__(self, keepalive_min, write_data_file=False, sync_remaining=False):
        self.keepalive_min = keepalive_min      # profile keepalive-timeout (live: 2)
        self.write_data_file = write_data_file  # FIX A: ESP writes /data/<MAC>.txt
        self.sync_remaining = sync_remaining    # FIX C: 1-min scheduler syncs remaining
        self.record = 0          # user record limit-uptime (minutes). NATIVE: static
        self.remaining = 0       # true remaining in active session (minutes)
        self.active = False
        self.mac_file = False    # does /data/<MAC>.txt exist on the router?
        self.sync_minute = 0     # scheduler tick counter

    # ---- RouterOS /ip hotspot semantics (verified on 6.49.17) ----
    def credit(self, minutes, mac=None):
        """ESP credits the user: record set to full limit (per-coin logic), session starts."""
        self.record = minutes
        self.remaining = minutes
        self.active = True
        if self.write_data_file and mac:
            # FIX A: /file/add name=hotspot/data/<MAC>.txt contents="P64593#sep/08 12:00:00"
            self.mac_file = True
            self.mac_voucher = mac
        self.sync_minute = 0

    def run_minute(self):
        """One minute passes. Scheduler (FIX C) runs once per minute."""
        if self.active:
            self.remaining -= 1
        self.sync_minute += 1
        if self.active and self.sync_remaining and self.sync_minute >= 1:
            # FIX C: while session active -> record = live session-time-left
            # Never runs in the same tick as ESP credit (ESP writes after /active/remove,
            # and this scheduler skips inactive sessions), so no write collision.
            self.record = max(self.remaining, 0)
            self.sync_minute = 0

    def phone_nap(self, nap_min):
        """Phone idles. Keepalive timeout kills the session after keepalive_min."""
        if nap_min > self.keepalive_min:
            self.active = False  # session dropped by hotspot server
        else:
            self.remaining -= nap_min  # survived the nap, still online
            if self.sync_remaining:
                self.record = max(self.remaining, 0)

    # ---- What the customer actually experiences ----
    def reconnect_experience(self):
        """Phone re-enables WiFi -> captive portal -> login.html behavior."""
        if self.active:
            return "SEAMLESS (still online, no login)"
        if self.record <= 0:
            return "INSERT COIN (time genuinely exhausted) -> correct"
        if self.mac_file:
            # core.js: GET /data/<MAC>.txt -> voucher -> auto-login (connectBtn.click())
            if self.sync_remaining:
                return f"SEAMLESS auto-login, remaining restored EXACTLY: {self.record} min"
            else:
                return f"SEAMLESS auto-login, but record still FULL: {self.record} min (time RESET -> revenue leak)"
        # No data file: page shows INSERT COIN button, voucher field empty
        return "LOGIN WALL: 'insert coin' shown, customer thinks time is LOST (it is not)"


def scenario(label, keepalive, nap, fix_a=False, fix_c=False, credit_min=120, nap_after=90):
    """One end-to-end journey. Returns (label, outcome)."""
    h = Hotspot(keepalive_min=keepalive, write_data_file=fix_a, sync_remaining=fix_c)
    h.credit(credit_min, mac="P64593")      # customer inserts coin, ESP credits
    for _ in range(nap_after):              # customer surfs nap_after minutes
        h.run_minute()
    h.phone_nap(nap)                        # phone idle/screen off for nap minutes
    return label, h.reconnect_experience()

print("model loaded - RouterOS semantics: static record, keepalive kill, /data auto-login")


: 

In [ ]:
# Cell 3 - BASELINE: current production state (keepalive=2m, no /data writer, no sync)
# Run the smoke tests BEFORE any patch and record pass/fail.
BASELINE_EXPECT = {
    "nap1m":   "SEAMLESS",          # short nap < keepalive: expect no problem
    "nap10m":  "SEAMLESS",          # long nap > keepalive: expect auto-login with remaining
}

print("=== BASELINE (current live config: keepalive=2m, no data file, no sync) ===")
results = {}
results["nap1m"]  = scenario("short nap (1m)",   keepalive=2, nap=1)[1]
results["nap10m"] = scenario("long nap (10m)",   keepalive=2, nap=10)[1]
results["fresh"]  = scenario("fresh insert, no nap", keepalive=2, nap=0)[1]

failures = 0
for k, v in results.items():
    expected = "SEAMLESS" if k != "fresh" else "SEAMLESS"
    ok = v.startswith(expected)
    print(f"  [{k:8}] {'PASS' if ok else 'FAIL'} -> {v}")
    failures += 0 if ok else 1

print(f"\nBASELINE: {3 - failures}/3 pass. The customer's reported bug is reproduced: "
      f"long nap -> '{results['nap10m']}'")


In [ ]:
# Cell 4 - APPLY THE CANDIDATE PATCH (in the model only!) and re-run the same tests
# FIX A: ESP writes /data/<MAC>.txt after credit   (fix_a=True)
# FIX B: keepalive-timeout 2m -> 15m                (keepalive=15)
# FIX C: 1-min scheduler syncs remaining -> record  (fix_c=True)
print("=== AFTER PATCH (data file ON, sync ON, keepalive=15m) ===")
patched = {}
patched["nap1m"]  = scenario("short nap (1m)",  keepalive=15, nap=1,  fix_a=True, fix_c=True)[1]
patched["nap10m"] = scenario("long nap (10m)",  keepalive=15, nap=10, fix_a=True, fix_c=True)[1]
patched["nap20m"] = scenario("very long nap (20m)", keepalive=15, nap=20, fix_a=True, fix_c=True)[1]

for k, v in patched.items():
    ok = "SEAMLESS" in v
    print(f"  [{k:8}] {'PASS' if ok else 'FAIL'} -> {v}")

# Assertions = the actual smoke test verdicts
assert "SEAMLESS" in patched["nap1m"],   "short nap must stay seamless"
assert "restored EXACTLY" in patched["nap10m"], "long nap must auto-login with exact remaining"
assert "restored EXACTLY" in patched["nap20m"], "very long nap must auto-login with exact remaining"
print("\nPATCHED: 3/3 PASS - no login wall, no time reset, no time loss.")


In [ ]:
# Cell 5 - EDGE CASES & REGRESSION CHECKS (still theory, before touching the box)

# E1: voucher fully consumed -> record 0 -> INSERT COIN is CORRECT behavior
h = Hotspot(keepalive_min=15, write_data_file=True, sync_remaining=True)
h.credit(2, mac="P64593")
for _ in range(3):
    h.run_minute()
h.phone_nap(20)
out = h.reconnect_experience()
assert "INSERT COIN" in out, out
print("E1 time-exhausted ->", out)

# E2: ESP extend-while-online (kick-first) must not collide with the 1-min sync scheduler.
# Ordering guarantee: scheduler only writes while session ACTIVE; ESP kick happens on
# /active/remove -> session inactive -> scheduler skips that tick -> no lost update.
h = Hotspot(keepalive_min=15, write_data_file=True, sync_remaining=True)
h.credit(60, mac="P64593")
for _ in range(10):
    h.run_minute()          # record synced to ~50
esp_kicked = False          # ESP: removeLiveSession -> extend record 50 -> 110
h.active = False
h.record += 60
h.active = True
h.remaining = 110
for _ in range(2):
    h.run_minute()          # scheduler resumes syncing the EXTENDED session
assert h.record <= 110 and h.record > 100, h.record
print("E2 kick-first extend + scheduler: no collision, record =", h.record, "min")

# E3: /data file FORMAT CONTRACT - login.html does data.split('#')[0]
# ESP will write:  name=hotspot/data/<MAC>.txt   contents="P64593#sep/08 12:00:00"
file_content = "P64593#sep/08 12:00:00"
voucher = file_content.split("#")[0]
assert voucher == "P64593"
print("E3 /data file parse contract OK ->", voucher)

# E4: no voucher ever saved for this MAC -> no file -> login wall is LEGITIMATE
h = Hotspot(keepalive_min=15, write_data_file=False, sync_remaining=True)
h.credit(120, mac=None)   # mac unknown (new device) -> nothing to auto-login with
h.phone_nap(20)
out = h.reconnect_experience()
assert "LOGIN WALL" in out
print("E4 new MAC, no saved voucher ->", out, "(correct: cannot auto-login a stranger)")

# E5: time RESET regression - with FIX A but WITHOUT FIX C, re-login gives FULL time
h = Hotspot(keepalive_min=15, write_data_file=True, sync_remaining=False)
h.credit(120, mac="P64593")
for _ in range(90):
    h.run_minute()
h.phone_nap(20)
out = h.reconnect_experience()
assert "RESET" in out, out
print("E5 proves FIX C is REQUIRED: without sync ->", out)

print("\nALL EDGE CASES PASS")


## Verdict (theory)

- **Baseline reproduces the reported bug**: keepalive=2m + no `/data/<MAC>.txt` (our ESP32 never writes it — old NodeMCU did) + static record => long nap => INSERT COIN wall, customer believes time is lost.
- **Patched model passes 3/3 + 5 edge cases**: FIX A (ESP writes `/data/<MAC>.txt` via `/file/add contents=`) + FIX B (keepalive 2m→15m) + FIX C (1-min remaining-sync scheduler, active-only) => every reconnect restores the EXACT remaining time with no login wall.
- **E5 proves FIX C is mandatory** — without the sync, auto-login restores the FULL original limit (time reset, revenue leak).
- **E2 proves no write collision** between the ESP kick-first crediting and the 1-minute scheduler (scheduler only writes active sessions; ESP writes after `/active/remove`).

### Approved patch sequence (only after this validation)
1. ESP firmware: after `performCredit` success, write `/data/<MAC>.txt` on the router via the existing API connection.
2. Router: `/ip hotspot user profile set coin-hotspot keepalive-timeout=15m` (and `default`).
3. Router: add 1-min scheduler `sync-remaining`:
   `/ip hotspot active` foreach -> `/ip hotspot user set <user> limit-uptime=<session-time-left>` (skip `default-trial`; no write when inactive).
4. Re-flash via OTA (marker: new log line `[DATA] wrote /data/<MAC>.txt`) and re-test with the tester phone.